# 🚀 BiMamba-NAPM: Bidirectional Mamba with Normal-Anchored Prototype Memory
### Weakly-Supervised Video Anomaly Detection (WS-VAD)
**Tác giả:** Huỳnh Quang Vũ (MSSV: 2351010247)  
**Giảng viên hướng dẫn:** TS. Lê Viết Tuấn  
**Đơn vị:** Khoa Công nghệ Thông tin, Trường Đại học Mở TP. Hồ Chí Minh  

---
### 🎯 Mục tiêu hiệu năng:
- **UCF-Crime:** ROC-AUC >= 90.0%  
- **XD-Violence:** Average Precision (AP) > 87.0%  
- **Phương thức:** Huấn luyện từ đầu (Train from Scratch 100%), không mượn checkpoint warm-start của DSANet.

## ⚙️ PHẦN 1: HUẤN LUYỆN TRÊN TẬP DỮ LIỆU UCF-CRIME

In [ ]:
# ========================= USER CONFIG: UCF-CRIME =========================
DATASET = "ucf"
MAX_EPOCHS = 15
UCF_BATCH_SIZE = 64        # 32 samples per GPU on 2x T4
LR = 5e-5                  # Optimal learning rate for Scratch training
NUM_WORKERS = 4
SEED = 2026
SSM_TYPE = "s4d"           # Fast parallel S4D state space model
REPO_URL = "https://github.com/Sharvuz/BiMamba-NAPM.git"


In [ ]:
# ========================= ENVIRONMENT SETUP =========================
import os, sys, shutil, subprocess, time, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = WORK / "BiMamba-NAPM"
ARTIFACTS = WORK / "bimamba_artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)

os.chdir(str(REPO))
print(f"Active Directory: {os.getcwd()}")


In [ ]:
# ========================= TRAIN BIMAMBA-NAPM (UCF-CRIME) =========================
dataset_dir = ARTIFACTS / "ucf"
dataset_dir.mkdir(parents=True, exist_ok=True)

train_cmd = [
    sys.executable, "-u", "src/ucf_train.py",
    "--dataset", "ucf",
    "--ssm_type", SSM_TYPE,
    "--num_layers", "2",
    "--num_prototypes", "16",
    "--batch_size", str(UCF_BATCH_SIZE),
    "--lr", str(LR),
    "--max_epoch", str(MAX_EPOCHS),
    "--loss_napm_weight", "0.5",
    "--loss_align_weight", "0.5",
    "--save_dir", str(dataset_dir)
]

print("Executing:", " ".join(train_cmd))
subprocess.run(train_cmd, check=True)


In [ ]:
# ========================= EVALUATION: UCF-CRIME =========================
best_model_path = ARTIFACTS / "ucf" / "bimamba_napm_ucf_best.pth"

eval_cmd = [
    sys.executable, "-u", "src/ucf_test.py",
    "--dataset", "ucf",
    "--ssm_type", SSM_TYPE,
    "--model_path", str(best_model_path)
]
print("Evaluating:", " ".join(eval_cmd))
subprocess.run(eval_cmd, check=True)


## ⚙️ PHẦN 2: HUẤN LUYỆN TRÊN TẬP DỮ LIỆU XD-VIOLENCE

In [ ]:
# ========================= USER CONFIG: XD-VIOLENCE =========================
DATASET_XD = "xd"
MAX_EPOCHS_XD = 20
XD_BATCH_SIZE = 64
LR_XD = 1e-4

dataset_xd_dir = ARTIFACTS / "xd"
dataset_xd_dir.mkdir(parents=True, exist_ok=True)

train_xd_cmd = [
    sys.executable, "-u", "src/xd_train.py",
    "--dataset", "xd",
    "--ssm_type", SSM_TYPE,
    "--num_layers", "2",
    "--num_prototypes", "16",
    "--batch_size", str(XD_BATCH_SIZE),
    "--lr", str(LR_XD),
    "--max_epoch", str(MAX_EPOCHS_XD),
    "--loss_napm_weight", "0.5",
    "--loss_align_weight", "0.5",
    "--save_dir", str(dataset_xd_dir)
]

print("Executing:", " ".join(train_xd_cmd))
subprocess.run(train_xd_cmd, check=True)


In [ ]:
# ========================= EVALUATION: XD-VIOLENCE =========================
best_model_xd = ARTIFACTS / "xd" / "bimamba_napm_xd_best.pth"

eval_xd_cmd = [
    sys.executable, "-u", "src/xd_test.py",
    "--dataset", "xd",
    "--ssm_type", SSM_TYPE,
    "--model_path", str(best_model_xd)
]
print("Evaluating:", " ".join(eval_xd_cmd))
subprocess.run(eval_xd_cmd, check=True)
